In [90]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import numpy as np

In [21]:
# headers = {
#     'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
#     'Accept-Language': 'en-US,en;q=0.9'}
# webpage=requests.get('https://www.imdb.com/chart/top/',headers=headers).text

In [117]:
from selenium import webdriver
driver = webdriver.Chrome()
driver.get("https://www.imdb.com/chart/top/")

In [41]:
html = driver.page_source

In [42]:
soup = BeautifulSoup(html, "html.parser")

# Fetching Data From the Single Page

In [85]:
rank = []
title = []
year = []
runtime = []
certificate = []
rating = []
voteCount = []
movies = soup.find_all("li", class_="ipc-metadata-list-summary-item")

for movie in movies:
    rank.append(movie.find("div", class_="ipc-signpost__text").text)
    title.append(movie.find("h4").text)

    year.append(movie.find_all("li", class_="ipc-inline-list__item")[0].text)
    runtime.append(movie.find_all("li", class_="ipc-inline-list__item")[1].text)
    try:
        certificate.append(movie.find_all("li", class_="ipc-inline-list__item")[2].text)
    except:
        certificate.append(np.nan)
    rating.append(movie.find("span", class_="ipc-rating-star--rating").text)
    voteCount.append(movie.find('span',class_='ipc-rating-star--voteCount').text.strip().strip('()'))
    

# Automatically Getting Director's Name and Generes

In [97]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

directors = []
genres = [] 

wait = WebDriverWait(driver, 5)

for i in range(250):
    button_xpath = f'(//button[@data-testid="title-summary-prompt-button-info-icon"])[{i+1}]'
    button = wait.until(EC.presence_of_element_located((By.XPATH, button_xpath)))
    
    driver.execute_script(
        "arguments[0].scrollIntoView({behavior:'instant', block:'center'});",
        button
    )
    
    wait.until(EC.element_to_be_clickable((By.XPATH, button_xpath))).click()

    try:
        director_xpath = (
            '//div[@data-testid="p_ct"]'
            '//div[@data-testid="c_ct"][.//span[text()="Director" or text()="Directors"]]'
            '//a'
        )
        director_element = wait.until(EC.visibility_of_element_located((By.XPATH, director_xpath)))
        directors.append(director_element.text)
    except Exception:
        directors.append("N/A") 

    try:
        genre_selector = 'ul[data-testid="btp_gl"] li'
        genre_elements = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, genre_selector)))
        genre_text = ", ".join([g.text for g in genre_elements])
        genres.append(genre_text)
    except Exception:
        genres.append("N/A")
        
    close = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'button[aria-label="Close Prompt"]')))
    close.click()

    wait.until(EC.invisibility_of_element_located((By.CSS_SELECTOR, 'button[aria-label="Close Prompt"]')))

# Converting into Data Frame

In [100]:
d={'rank':rank,
   'title':title ,
   'year':year ,
   'runtime':runtime ,
   'certificate':certificate ,
   'rating':rating,
   'voteCount':voteCount,
   'director':director,
   'genre':genres}
df=pd.DataFrame(d)

# Displaying Data

In [101]:
df

,rank,title,year,runtime,certificate,rating,voteCount,director,genre
0,#1,The Shawshank Redemption,1994,2h 22m,R,9.3,3.2M,Frank Darabont,Drama
1,#2,The Godfather,1972,2h 55m,R,9.2,2.2M,Francis Ford Coppola,"Crime, Drama"
2,#3,The Dark Knight,2008,2h 32m,PG-13,9.1,3.2M,Christopher Nolan,"Crime, Thriller"
3,#4,The Godfather Part II,1974,3h 22m,R,9.0,1.5M,Francis Ford Coppola,"Crime, Drama"
4,#5,The Lord of the Rings: The Return of the King,2003,3h 21m,PG-13,9.0,2.2M,Peter Jackson,"Adventure, Drama, Fantasy"
...,...,...,...,...,...,...,...,...,...
245,#246,My Father and My Son,2005,1h 52m,Not Rated,8.2,102K,Çagan Irmak,Drama
246,#247,The Passion of Joan of Arc,1928,1h 54m,Passed,8.1,69K,Carl Theodor Dreyer,"Biography, Drama, History"
247,#248,The Handmaiden,2016,2h 25m,Not Rated,8.1,206K,Park Chan-wook,"Drama, Romance, Thriller"
248,#249,Sita Ramam,2022,2h 43m,NaN,8.5,85K,Hanu Raghavapudi,"Action, Drama, Musical"


# Saving Data into CSV Format

In [102]:
df.to_csv('Top 250 Movies.csv')